# 📊 Advanced Graph Reasoning - Comprehensive Evaluation

This notebook provides **thorough evaluation** of the 5 advanced reasoning query types.

## Evaluation Components:
1. **Quantitative Analysis** - Success rates, coverage, timing metrics
2. **Qualitative Analysis** - Output quality, reasoning trace correctness
3. **Failure Case Analysis** - Systematic detection of edge cases and failures

## Query Types Evaluated:
1. Chain Reasoning (Multi-hop)
2. Pattern Matching (Subgraph)
3. Scene Comparison
4. Counterfactual Reasoning
5. Centrality Queries

In [1]:
# Setup and Imports
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Any, Tuple
import json

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'advanced_queries' else Path.cwd()
sys.path.insert(0, str(project_root))

from advanced_queries.advanced_reasoning_engine import AdvancedReasoningEngine, AdvancedReasoningResult

print("✅ Imports successful!")

✅ Imports successful!


In [2]:
# Initialize Engine
engine = AdvancedReasoningEngine(scale='10k')

# Get basic stats
print(f"\n📊 Graph Statistics:")
print(f"   Nodes: {engine.graph.number_of_nodes():,}")
print(f"   Edges: {engine.graph.number_of_edges():,}")
print(f"   Concepts: {len(engine._concept_nodes):,}")
print(f"   Attributes: {len(engine._attribute_nodes):,}")
print(f"   Images: {len(engine._image_to_objects):,}")
print(f"   Relation Types: {len(engine._relation_index):,}")

📂 Loading graph from: d:\Giáo trình 20251\IT3930E - Project III\hybrid_multimodal_retrieval\experiments\sample_10k\gqa_lightrag.gpickle
   ✅ Đã load thành công!
   📊 Nodes: 164,585
   🔗 Edges: 738,736
   🔧 Building cache...
   ✅ Cache đã sẵn sàng!
      • Instance nodes: 162,453
      • Concept nodes: 1,525
      • Attribute nodes: 607
      • Images: 9,901
   🔧 Building advanced reasoning cache...
      • Relation types indexed: 291
   ✅ Advanced cache ready!

📊 Graph Statistics:
   Nodes: 164,585
   Edges: 738,736
   Concepts: 1,525
   Attributes: 607
   Images: 9,901
   Relation Types: 291


In [3]:
# Helper functions for evaluation

class EvaluationMetrics:
    """Track evaluation metrics for a query type."""
    def __init__(self, query_type: str):
        self.query_type = query_type
        self.total_queries = 0
        self.successful_queries = 0  # Returned at least 1 result
        self.empty_results = 0
        self.errors = 0
        self.total_time = 0.0
        self.result_counts = []
        self.failure_cases = []
        
    def record(self, result: AdvancedReasoningResult, elapsed: float, query_params: Dict):
        self.total_queries += 1
        self.total_time += elapsed
        
        if isinstance(result.results, list):
            count = len(result.results)
        elif isinstance(result.results, dict):
            count = 1 if result.results else 0
        else:
            count = 1 if result.results else 0
        
        self.result_counts.append(count)
        
        if count > 0:
            self.successful_queries += 1
        else:
            self.empty_results += 1
            self.failure_cases.append({
                'params': query_params,
                'reason': 'empty_result',
                'metadata': result.metadata
            })
    
    def record_error(self, error: Exception, query_params: Dict):
        self.total_queries += 1
        self.errors += 1
        self.failure_cases.append({
            'params': query_params,
            'reason': 'exception',
            'error': str(error)
        })
    
    def summary(self) -> Dict:
        avg_time = self.total_time / max(1, self.total_queries)
        avg_results = sum(self.result_counts) / max(1, len(self.result_counts))
        success_rate = self.successful_queries / max(1, self.total_queries)
        
        return {
            'query_type': self.query_type,
            'total_queries': self.total_queries,
            'successful': self.successful_queries,
            'empty_results': self.empty_results,
            'errors': self.errors,
            'success_rate': f"{success_rate:.1%}",
            'avg_time_ms': f"{avg_time*1000:.1f}",
            'avg_result_count': f"{avg_results:.1f}",
            'failure_count': len(self.failure_cases)
        }
    
    def print_summary(self):
        s = self.summary()
        print(f"\n{'='*60}")
        print(f"📊 {s['query_type']} Evaluation Summary")
        print(f"{'='*60}")
        print(f"   Total Queries: {s['total_queries']}")
        print(f"   Successful: {s['successful']} ({s['success_rate']})")
        print(f"   Empty Results: {s['empty_results']}")
        print(f"   Errors: {s['errors']}")
        print(f"   Avg Time: {s['avg_time_ms']} ms")
        print(f"   Avg Results: {s['avg_result_count']}")

print("✅ Evaluation helpers ready!")

✅ Evaluation helpers ready!


---
# 🔗 1. Chain Reasoning Evaluation

Testing multi-hop chain traversal with various:
- Chain lengths (1-hop, 2-hop, 3-hop)
- Concept combinations (common vs rare)
- Attribute constraints (with/without)

In [4]:
# Get top concepts for testing
concept_counts = {}
for concept_id in engine._concept_nodes:
    name = concept_id.replace('Concept:', '')
    count = len(engine._concept_to_instances.get(concept_id, set()))
    concept_counts[name] = count

sorted_concepts = sorted(concept_counts.items(), key=lambda x: -x[1])
top_concepts = [c[0] for c in sorted_concepts[:30]]
rare_concepts = [c[0] for c in sorted_concepts[-30:] if c[1] > 0]

print("Top 10 concepts:", top_concepts[:10])
print("\nSome rare concepts:", rare_concepts[:10])

Top 10 concepts: ['window', 'man', 'shirt', 'tree', 'wall', 'person', 'building', 'ground', 'sky', 'head']

Some rare concepts: ['brownies', 'ravioli', 'snow shoes', 'wolves', 'stir fry', 'customers', 'lobby', 'shoppers', 'guacamole', 'bartender']


In [5]:
# Get common relations
relation_counts = {rel: len(edges) for rel, edges in engine._relation_index.items()}
sorted_relations = sorted(relation_counts.items(), key=lambda x: -x[1])
top_relations = [r[0] for r in sorted_relations[:20]]

print("Top 10 relations:")
for rel, count in sorted_relations[:10]:
    print(f"   {rel}: {count:,} edges")

Top 10 relations:
   to the right of: 219,249 edges
   to the left of: 219,235 edges
   on: 10,994 edges
   wearing: 6,808 edges
   of: 4,698 edges
   in: 3,792 edges
   near: 3,364 edges
   behind: 2,146 edges
   in front of: 2,144 edges
   on top of: 1,174 edges


In [6]:
# Chain Reasoning: Quantitative Evaluation
chain_metrics = EvaluationMetrics("Chain Reasoning")

# Test cases
chain_test_cases = [
    # 1-hop chains (should have high success)
    {"start": "person", "chain": [{"relation": "wearing", "concept": "shirt"}]},
    {"start": "person", "chain": [{"relation": "wearing", "concept": "hat"}]},
    {"start": "man", "chain": [{"relation": "wearing", "concept": "jacket"}]},
    {"start": "woman", "chain": [{"relation": "wearing", "concept": "dress"}]},
    {"start": "dog", "chain": [{"relation": "on", "concept": "grass"}]},
    
    # 2-hop chains (medium success)
    {"start": "person", "chain": [{"relation": "wearing", "concept": "shirt"}, {"relation": "to the left of", "concept": "tree"}]},
    {"start": "person", "chain": [{"relation": "holding"}, {"relation": "on", "concept": "table"}]},
    {"start": "man", "chain": [{"relation": "wearing", "concept": "shirt"}, {"relation": "to the right of", "concept": "woman"}]},
    {"start": "car", "chain": [{"relation": "on", "concept": "road"}, {"relation": "to the left of", "concept": "building"}]},
    {"start": "person", "chain": [{"relation": "sitting on"}, {"relation": "in", "concept": "room"}]},
    
    # Chains with attribute constraints (lower success expected)
    {"start": "person", "chain": [{"relation": "wearing", "concept": "shirt", "attribute": "white"}]},
    {"start": "person", "chain": [{"relation": "wearing", "concept": "shirt", "attribute": "red"}]},
    {"start": "car", "chain": [{"relation": "on", "concept": "road", "attribute": "black"}]},
    
    # Rare concept chains (likely to fail)
    {"start": "elephant", "chain": [{"relation": "in", "concept": "zoo"}]},
    {"start": "airplane", "chain": [{"relation": "in", "concept": "sky"}]},
    
    # 3-hop chains (likely to fail or have few results)
    {"start": "person", "chain": [{"relation": "wearing", "concept": "shirt"}, {"relation": "to the left of", "concept": "table"}, {"relation": "in", "concept": "room"}]},
]

print("Running Chain Reasoning evaluation...")
for i, tc in enumerate(chain_test_cases):
    try:
        start_time = time.time()
        result = engine.chain_reasoning(
            start_concept=tc["start"],
            chain=tc["chain"],
            limit=20
        )
        elapsed = time.time() - start_time
        chain_metrics.record(result, elapsed, tc)
        
        status = "✅" if result.results else "❌"
        count = len(result.results) if isinstance(result.results, list) else 0
        print(f"   {status} Test {i+1}: {tc['start']} → {len(tc['chain'])}-hop → {count} results ({elapsed*1000:.1f}ms)")
        
    except Exception as e:
        chain_metrics.record_error(e, tc)
        print(f"   ❌ Test {i+1}: ERROR - {str(e)[:50]}")

chain_metrics.print_summary()

Running Chain Reasoning evaluation...
   ✅ Test 1: person → 1-hop → 20 results (9.9ms)
   ✅ Test 2: person → 1-hop → 13 results (11.0ms)
   ✅ Test 3: man → 1-hop → 20 results (9.9ms)
   ✅ Test 4: woman → 1-hop → 10 results (8.1ms)
   ✅ Test 5: dog → 1-hop → 8 results (3.0ms)
   ❌ Test 6: person → 2-hop → 0 results (7.2ms)
   ❌ Test 7: person → 2-hop → 0 results (6.1ms)
   ✅ Test 8: man → 2-hop → 6 results (7.7ms)
   ✅ Test 9: car → 2-hop → 4 results (6.8ms)
   ❌ Test 10: person → 2-hop → 0 results (7.2ms)
   ✅ Test 11: person → 1-hop → 8 results (7.9ms)
   ✅ Test 12: person → 1-hop → 1 results (5.9ms)
   ❌ Test 13: car → 1-hop → 0 results (4.7ms)
   ✅ Test 14: elephant → 1-hop → 5 results (4.7ms)
   ✅ Test 15: airplane → 1-hop → 20 results (3.5ms)
   ❌ Test 16: person → 3-hop → 0 results (7.8ms)

📊 Chain Reasoning Evaluation Summary
   Total Queries: 16
   Successful: 11 (68.8%)
   Empty Results: 5
   Errors: 0
   Avg Time: 7.0 ms
   Avg Results: 7.2


In [7]:
# Chain Reasoning: Failure Case Analysis
print("\n🔍 CHAIN REASONING - FAILURE CASE ANALYSIS")
print("="*60)

if chain_metrics.failure_cases:
    # Categorize failures
    failure_categories = defaultdict(list)
    
    for fc in chain_metrics.failure_cases:
        params = fc['params']
        chain_len = len(params.get('chain', []))
        has_attr = any('attribute' in hop for hop in params.get('chain', []))
        
        if chain_len >= 3:
            failure_categories['long_chain'].append(fc)
        elif has_attr:
            failure_categories['attribute_constraint'].append(fc)
        else:
            failure_categories['concept_mismatch'].append(fc)
    
    print(f"\nFailure Categories:")
    for cat, cases in failure_categories.items():
        print(f"\n  📌 {cat.replace('_', ' ').title()}: {len(cases)} cases")
        for case in cases[:2]:  # Show first 2 examples
            print(f"      • Start: {case['params'].get('start')}")
            print(f"        Chain: {case['params'].get('chain')}")
else:
    print("No failures detected!")


🔍 CHAIN REASONING - FAILURE CASE ANALYSIS

Failure Categories:

  📌 Concept Mismatch: 3 cases
      • Start: person
        Chain: [{'relation': 'wearing', 'concept': 'shirt'}, {'relation': 'to the left of', 'concept': 'tree'}]
      • Start: person
        Chain: [{'relation': 'holding'}, {'relation': 'on', 'concept': 'table'}]

  📌 Attribute Constraint: 1 cases
      • Start: car
        Chain: [{'relation': 'on', 'concept': 'road', 'attribute': 'black'}]

  📌 Long Chain: 1 cases
      • Start: person
        Chain: [{'relation': 'wearing', 'concept': 'shirt'}, {'relation': 'to the left of', 'concept': 'table'}, {'relation': 'in', 'concept': 'room'}]


In [8]:
# Chain Reasoning: Qualitative Example
print("\n📝 CHAIN REASONING - QUALITATIVE ANALYSIS")
print("="*60)

# Run a successful query with full output
result = engine.chain_reasoning(
    start_concept="person",
    chain=[{"relation": "wearing", "concept": "shirt"}],
    limit=3
)

print("\n✅ Successful Query Example:")
result.print_result(verbose=True)

# Check reasoning trace quality
print("\n📋 Reasoning Trace Quality Check:")
for step in result.reasoning_steps:
    has_operation = bool(step.operation)
    has_description = bool(step.description)
    has_io = step.input_size >= 0 and step.output_size >= 0
    quality = "✅" if (has_operation and has_description and has_io) else "⚠️"
    print(f"   {quality} Step {step.step_number}: op={has_operation}, desc={has_description}, io={has_io}")


📝 CHAIN REASONING - QUALITATIVE ANALYSIS

✅ Successful Query Example:

🧠 ADVANCED QUERY: 1. Chain Reasoning (Multi-Hop)

❓ QUESTION: Find images with 'person' → wearing shirt

🔍 REASONING STEPS (4 steps):
------------------------------------------------------------
   Step 1: 📥 [RETRIEVE]
            Get all instances of concept 'person'
            📊 0 → 2,786 items
            • concept: person

   Step 2: 🔗 [TRAVERSE]
            Follow 'wearing' edges from current nodes
            📊 500 → 148 items
            • relation: wearing
            • edges_checked: 148

   Step 3: 🔍 [FILTER]
            Keep only nodes of concept 'shirt'
            📊 148 → 29 items
            • concept: shirt
            • concept_instances: 3301

   Step 4: 📊 [AGGREGATE]
            Group valid chains by image and deduplicate
            📊 29 → 3 items
            • unique_images: 3

📊 RESULTS:
   1. image_id=2395187, chain_path=['2395187:4390019', '--[wearing]-->', '2395187:4389992'], chain_length=2

---
# 🎯 2. Pattern Matching Evaluation

Testing subgraph pattern matching with:
- Simple patterns (2 nodes, 1 edge)
- Complex patterns (3+ nodes, multiple edges)
- Star patterns (one node with multiple relations)

In [9]:
# Pattern Matching: Quantitative Evaluation
pattern_metrics = EvaluationMetrics("Pattern Matching")

pattern_test_cases = [
    # Simple 2-node patterns (high success)
    {"nodes": ["person", "shirt"], "edges": [("person", "wearing", "shirt")]},
    {"nodes": ["person", "hat"], "edges": [("person", "wearing", "hat")]},
    {"nodes": ["man", "woman"], "edges": [("man", "to the left of", "woman")]},
    {"nodes": ["car", "road"], "edges": [("car", "on", "road")]},
    {"nodes": ["tree", "grass"], "edges": [("tree", "on", "grass")]},
    
    # Star patterns (one concept with multiple relations)
    {"nodes": ["person", "shirt", "hat"], "edges": [("person", "wearing", "shirt"), ("person", "wearing", "hat")]},
    {"nodes": ["person", "shirt", "pant"], "edges": [("person", "wearing", "shirt"), ("person", "wearing", "pant")]},
    {"nodes": ["man", "woman", "child"], "edges": [("man", "to the left of", "woman"), ("woman", "to the left of", "child")]},
    
    # Chain patterns
    {"nodes": ["person", "table", "chair"], "edges": [("person", "to the left of", "table"), ("table", "to the left of", "chair")]},
    {"nodes": ["car", "tree", "building"], "edges": [("car", "to the left of", "tree"), ("tree", "to the left of", "building")]},
    
    # Rare patterns (likely to fail)
    {"nodes": ["elephant", "zebra"], "edges": [("elephant", "to the left of", "zebra")]},
    {"nodes": ["airplane", "bird"], "edges": [("airplane", "above", "bird")]},
    
    # Complex patterns (3+ edges)
    {"nodes": ["person", "shirt", "hat", "shoe"], 
     "edges": [("person", "wearing", "shirt"), ("person", "wearing", "hat"), ("person", "wearing", "shoe")]},
]

print("Running Pattern Matching evaluation...")
for i, tc in enumerate(pattern_test_cases):
    try:
        start_time = time.time()
        result = engine.pattern_matching(
            pattern_nodes=tc["nodes"],
            pattern_edges=tc["edges"],
            limit=20
        )
        elapsed = time.time() - start_time
        pattern_metrics.record(result, elapsed, tc)
        
        status = "✅" if result.results else "❌"
        count = len(result.results) if isinstance(result.results, list) else 0
        print(f"   {status} Test {i+1}: {tc['nodes']} → {count} results ({elapsed*1000:.1f}ms)")
        
    except Exception as e:
        pattern_metrics.record_error(e, tc)
        print(f"   ❌ Test {i+1}: ERROR - {str(e)[:50]}")

pattern_metrics.print_summary()

Running Pattern Matching evaluation...
   ✅ Test 1: ['person', 'shirt'] → 20 results (11.4ms)
   ✅ Test 2: ['person', 'hat'] → 20 results (8.0ms)
   ✅ Test 3: ['man', 'woman'] → 20 results (7.2ms)
   ✅ Test 4: ['car', 'road'] → 20 results (4.5ms)
   ✅ Test 5: ['tree', 'grass'] → 6 results (21.1ms)
   ✅ Test 6: ['person', 'shirt', 'hat'] → 12 results (11.8ms)
   ❌ Test 7: ['person', 'shirt', 'pant'] → 0 results (6.5ms)
   ✅ Test 8: ['man', 'woman', 'child'] → 14 results (9.6ms)
   ✅ Test 9: ['person', 'table', 'chair'] → 8 results (8.0ms)
   ✅ Test 10: ['car', 'tree', 'building'] → 20 results (13.4ms)
   ✅ Test 11: ['elephant', 'zebra'] → 1 results (0.7ms)
   ❌ Test 12: ['airplane', 'bird'] → 0 results (1.0ms)
   ✅ Test 13: ['person', 'shirt', 'hat', 'shoe'] → 1 results (10.0ms)

📊 Pattern Matching Evaluation Summary
   Total Queries: 13
   Successful: 11 (84.6%)
   Empty Results: 2
   Errors: 0
   Avg Time: 8.7 ms
   Avg Results: 10.9


In [10]:
# Pattern Matching: Failure Case Analysis
print("\n🔍 PATTERN MATCHING - FAILURE CASE ANALYSIS")
print("="*60)

if pattern_metrics.failure_cases:
    failure_categories = defaultdict(list)
    
    for fc in pattern_metrics.failure_cases:
        params = fc['params']
        num_edges = len(params.get('edges', []))
        nodes = params.get('nodes', [])
        
        # Check if nodes are rare
        rare_nodes = [n for n in nodes if concept_counts.get(n, 0) < 50]
        
        if rare_nodes:
            failure_categories['rare_concepts'].append(fc)
        elif num_edges >= 3:
            failure_categories['complex_pattern'].append(fc)
        else:
            failure_categories['relation_not_found'].append(fc)
    
    print(f"\nFailure Categories:")
    for cat, cases in failure_categories.items():
        print(f"\n  📌 {cat.replace('_', ' ').title()}: {len(cases)} cases")
        for case in cases[:2]:
            print(f"      • Nodes: {case['params'].get('nodes')}")
            print(f"        Edges: {case['params'].get('edges')}")
else:
    print("No failures detected!")


🔍 PATTERN MATCHING - FAILURE CASE ANALYSIS

Failure Categories:

  📌 Rare Concepts: 1 cases
      • Nodes: ['person', 'shirt', 'pant']
        Edges: [('person', 'wearing', 'shirt'), ('person', 'wearing', 'pant')]

  📌 Relation Not Found: 1 cases
      • Nodes: ['airplane', 'bird']
        Edges: [('airplane', 'above', 'bird')]


In [11]:
# Pattern Matching: Qualitative Example
print("\n📝 PATTERN MATCHING - QUALITATIVE ANALYSIS")
print("="*60)

result = engine.pattern_matching(
    pattern_nodes=["person", "shirt", "hat"],
    pattern_edges=[
        ("person", "wearing", "shirt"),
        ("person", "wearing", "hat")
    ],
    limit=3
)

print("\n✅ Star Pattern Example (person wearing shirt AND hat):")
result.print_result(verbose=True)


📝 PATTERN MATCHING - QUALITATIVE ANALYSIS

✅ Star Pattern Example (person wearing shirt AND hat):

🧠 ADVANCED QUERY: 2. Subgraph Pattern Matching

❓ QUESTION: Find images matching pattern: (person wearing shirt) + (person wearing hat)

🔍 REASONING STEPS (5 steps):
------------------------------------------------------------
   Step 1: 📥 [RETRIEVE]
            Get instances of concept 'person'
            📊 0 → 2,786 items
            • concept: person

   Step 2: 📥 [RETRIEVE]
            Get instances of concept 'shirt'
            📊 0 → 3,301 items
            • concept: shirt

   Step 3: 📥 [RETRIEVE]
            Get instances of concept 'hat'
            📊 0 → 876 items
            • concept: hat

   Step 4: 🔍 [FILTER]
            Keep images with all required concepts
            📊 3,366 → 101 items
            • required_concepts: ['person', 'shirt', 'hat']

   Step 5: 🎯 [MATCH]
            Verify pattern edges exist in each image
            📊 101 → 3 items
            • pattern_

---
# ⚖️ 3. Scene Comparison Evaluation

Testing scene comparison with:
- Similar scenes (same concepts)
- Different scenes (disjoint concepts)
- Edge cases (empty images, single object)

In [12]:
# Prepare test images with different characteristics
image_stats = []
for img_id in list(engine._image_to_objects.keys())[:500]:
    objects = engine._image_to_objects[img_id]
    concepts = set()
    for obj_id in objects:
        info = engine._get_node_info(obj_id)
        concepts.add(info.get('name', ''))
    
    image_stats.append({
        'image_id': img_id,
        'object_count': len(objects),
        'concept_count': len(concepts),
        'concepts': concepts
    })

# Sort by object count
image_stats.sort(key=lambda x: x['object_count'])

small_images = [img for img in image_stats if img['object_count'] <= 5][:5]
medium_images = [img for img in image_stats if 10 <= img['object_count'] <= 20][:5]
large_images = [img for img in image_stats if img['object_count'] >= 30][:5]

print(f"Small images (<= 5 objects): {len(small_images)}")
print(f"Medium images (10-20 objects): {len(medium_images)}")
print(f"Large images (>= 30 objects): {len(large_images)}")

Small images (<= 5 objects): 5
Medium images (10-20 objects): 5
Large images (>= 30 objects): 5


In [13]:
# Scene Comparison: Quantitative Evaluation
scene_metrics = EvaluationMetrics("Scene Comparison")

# Find image pairs with high/low concept overlap
def find_similar_pair():
    for i, img1 in enumerate(image_stats[:100]):
        for img2 in image_stats[i+1:100]:
            overlap = len(img1['concepts'] & img2['concepts'])
            if overlap >= 3:
                return img1['image_id'], img2['image_id']
    return None, None

def find_different_pair():
    for i, img1 in enumerate(image_stats[:100]):
        for img2 in image_stats[i+1:100]:
            overlap = len(img1['concepts'] & img2['concepts'])
            if overlap == 0 and img1['object_count'] > 3 and img2['object_count'] > 3:
                return img1['image_id'], img2['image_id']
    return None, None

similar_pair = find_similar_pair()
different_pair = find_different_pair()

print(f"Similar pair: {similar_pair}")
print(f"Different pair: {different_pair}")

Similar pair: ('2329764', '2332842')
Different pair: ('2331245', '2388456')


In [14]:
# Build test cases
scene_test_cases = []

# Random pairs
random.seed(42)
for _ in range(10):
    idx1, idx2 = random.sample(range(len(image_stats)), 2)
    scene_test_cases.append({
        'image1': image_stats[idx1]['image_id'],
        'image2': image_stats[idx2]['image_id'],
        'type': 'random'
    })

# Small image pairs (potential edge case)
if len(small_images) >= 2:
    scene_test_cases.append({
        'image1': small_images[0]['image_id'],
        'image2': small_images[1]['image_id'],
        'type': 'small'
    })

# Large image pairs
if len(large_images) >= 2:
    scene_test_cases.append({
        'image1': large_images[0]['image_id'],
        'image2': large_images[1]['image_id'],
        'type': 'large'
    })

# Similar pair
if similar_pair[0]:
    scene_test_cases.append({
        'image1': similar_pair[0],
        'image2': similar_pair[1],
        'type': 'similar'
    })

# Different pair
if different_pair[0]:
    scene_test_cases.append({
        'image1': different_pair[0],
        'image2': different_pair[1],
        'type': 'different'
    })

print(f"\nRunning Scene Comparison evaluation ({len(scene_test_cases)} tests)...")

similarity_scores = []
for i, tc in enumerate(scene_test_cases):
    try:
        start_time = time.time()
        result = engine.scene_comparison(
            image_id_1=tc['image1'],
            image_id_2=tc['image2']
        )
        elapsed = time.time() - start_time
        scene_metrics.record(result, elapsed, tc)
        
        sim_score = result.results.get('comparison', {}).get('overall_similarity', 0)
        similarity_scores.append((tc['type'], sim_score))
        print(f"   ✅ Test {i+1} [{tc['type']}]: similarity={sim_score:.3f} ({elapsed*1000:.1f}ms)")
        
    except Exception as e:
        scene_metrics.record_error(e, tc)
        print(f"   ❌ Test {i+1}: ERROR - {str(e)[:50]}")

scene_metrics.print_summary()


Running Scene Comparison evaluation (14 tests)...
   ✅ Test 1 [random]: similarity=0.267 (0.2ms)
   ✅ Test 2 [random]: similarity=0.056 (0.1ms)
   ✅ Test 3 [random]: similarity=0.224 (0.1ms)
   ✅ Test 4 [random]: similarity=0.244 (0.1ms)
   ✅ Test 5 [random]: similarity=0.352 (0.1ms)
   ✅ Test 6 [random]: similarity=0.205 (0.1ms)
   ✅ Test 7 [random]: similarity=0.292 (0.2ms)
   ✅ Test 8 [random]: similarity=0.167 (0.1ms)
   ✅ Test 9 [random]: similarity=0.083 (0.1ms)
   ✅ Test 10 [random]: similarity=0.000 (0.0ms)
   ✅ Test 11 [small]: similarity=0.000 (0.0ms)
   ✅ Test 12 [large]: similarity=0.283 (0.3ms)
   ✅ Test 13 [similar]: similarity=0.500 (0.1ms)
   ✅ Test 14 [different]: similarity=0.000 (0.0ms)

📊 Scene Comparison Evaluation Summary
   Total Queries: 14
   Successful: 14 (100.0%)
   Empty Results: 0
   Errors: 0
   Avg Time: 0.1 ms
   Avg Results: 1.0


In [15]:
# Scene Comparison: Analyze similarity distribution
print("\n📊 SCENE COMPARISON - SIMILARITY DISTRIBUTION")
print("="*60)

by_type = defaultdict(list)
for type_name, score in similarity_scores:
    by_type[type_name].append(score)

for type_name, scores in by_type.items():
    avg = sum(scores) / len(scores)
    min_s = min(scores)
    max_s = max(scores)
    print(f"   {type_name}: avg={avg:.3f}, min={min_s:.3f}, max={max_s:.3f}")


📊 SCENE COMPARISON - SIMILARITY DISTRIBUTION
   random: avg=0.189, min=0.000, max=0.352
   small: avg=0.000, min=0.000, max=0.000
   large: avg=0.283, min=0.283, max=0.283
   similar: avg=0.500, min=0.500, max=0.500
   different: avg=0.000, min=0.000, max=0.000


In [16]:
# Scene Comparison: Qualitative Example
print("\n📝 SCENE COMPARISON - QUALITATIVE ANALYSIS")
print("="*60)

if similar_pair[0]:
    result = engine.scene_comparison(similar_pair[0], similar_pair[1])
    print("\n✅ Similar Scenes Comparison:")
    result.print_result(verbose=True)


📝 SCENE COMPARISON - QUALITATIVE ANALYSIS

✅ Similar Scenes Comparison:

🧠 ADVANCED QUERY: 3. Scene Comparison

❓ QUESTION: Compare scenes: image '2329764' vs image '2332842'

🔍 REASONING STEPS (4 steps):
------------------------------------------------------------
   Step 1: 📥 [RETRIEVE]
            Extract scene graph for image '2329764'
            📊 0 → 3 items
            • objects: 3
            • concepts: 3
            • attributes: 2

   Step 2: 📥 [RETRIEVE]
            Extract scene graph for image '2332842'
            📊 0 → 7 items
            • objects: 7
            • concepts: 6
            • attributes: 2

   Step 3: 🧮 [COMPUTE]
            Calculate structural similarity metrics
            📊 9 → 2 items
            • concept_jaccard: 0.5
            • attribute_jaccard: 1.0

   Step 4: ⚖️ [COMPARE]
            Compare relationship types between images
            📊 3 → 0 items
            • relation_types_1: 0
            • relation_types_2: 3
            • common_re

---
# 🔮 4. Counterfactual Reasoning Evaluation

Testing "what if" scenarios:
- Removing common concepts
- Removing rare concepts (edge case)
- Adding concepts that typically interact
- Adding unrelated concepts

In [17]:
# Counterfactual Reasoning: Quantitative Evaluation
counterfactual_metrics = EvaluationMetrics("Counterfactual Reasoning")

# Get sample images for testing
test_images = [img['image_id'] for img in medium_images[:5]]

counterfactual_test_cases = []

# Test removal of common concepts
for img_id in test_images[:3]:
    counterfactual_test_cases.append({
        'image_id': img_id,
        'remove_concept': 'person',
        'type': 'remove_common'
    })

# Test removal of concepts not in image (edge case)
for img_id in test_images[:2]:
    counterfactual_test_cases.append({
        'image_id': img_id,
        'remove_concept': 'elephant',  # Likely not in image
        'type': 'remove_missing'
    })

# Test adding common concepts
for img_id in test_images[:3]:
    counterfactual_test_cases.append({
        'image_id': img_id,
        'add_concept': 'dog',
        'type': 'add_common'
    })

# Test adding rare concepts
for img_id in test_images[:2]:
    counterfactual_test_cases.append({
        'image_id': img_id,
        'add_concept': 'giraffe',
        'type': 'add_rare'
    })

print(f"Running Counterfactual Reasoning evaluation ({len(counterfactual_test_cases)} tests)...")

for i, tc in enumerate(counterfactual_test_cases):
    try:
        start_time = time.time()
        result = engine.counterfactual_reasoning(
            image_id=tc['image_id'],
            remove_concept=tc.get('remove_concept'),
            add_concept=tc.get('add_concept')
        )
        elapsed = time.time() - start_time
        counterfactual_metrics.record(result, elapsed, tc)
        
        action = tc.get('remove_concept') or tc.get('add_concept')
        print(f"   ✅ Test {i+1} [{tc['type']}]: {action} ({elapsed*1000:.1f}ms)")
        
    except Exception as e:
        counterfactual_metrics.record_error(e, tc)
        print(f"   ❌ Test {i+1}: ERROR - {str(e)[:50]}")

counterfactual_metrics.print_summary()

Running Counterfactual Reasoning evaluation (10 tests)...
   ✅ Test 1 [remove_common]: person (194.9ms)
   ✅ Test 2 [remove_common]: person (162.5ms)
   ✅ Test 3 [remove_common]: person (158.6ms)
   ✅ Test 4 [remove_missing]: elephant (163.7ms)
   ✅ Test 5 [remove_missing]: elephant (171.3ms)
   ✅ Test 6 [add_common]: dog (2.2ms)
   ✅ Test 7 [add_common]: dog (1.3ms)
   ✅ Test 8 [add_common]: dog (1.0ms)
   ✅ Test 9 [add_rare]: giraffe (5.1ms)
   ✅ Test 10 [add_rare]: giraffe (3.4ms)

📊 Counterfactual Reasoning Evaluation Summary
   Total Queries: 10
   Successful: 10 (100.0%)
   Empty Results: 0
   Errors: 0
   Avg Time: 86.4 ms
   Avg Results: 1.0


In [18]:
# Counterfactual: Failure Case Analysis
print("\n🔍 COUNTERFACTUAL REASONING - FAILURE CASE ANALYSIS")
print("="*60)

# Test specific edge cases
edge_case_tests = [
    # Non-existent image
    {'image_id': 'NONEXISTENT_IMAGE', 'remove_concept': 'person'},
    # Both remove and add same concept
    {'image_id': test_images[0], 'remove_concept': 'tree', 'add_concept': 'tree'},
]

print("\nEdge Case Tests:")
for tc in edge_case_tests:
    try:
        result = engine.counterfactual_reasoning(**tc)
        print(f"   • {tc}: Returned result (check content)")
    except Exception as e:
        print(f"   • {tc}: Exception - {str(e)[:60]}")


🔍 COUNTERFACTUAL REASONING - FAILURE CASE ANALYSIS

Edge Case Tests:
   • {'image_id': 'NONEXISTENT_IMAGE', 'remove_concept': 'person'}: Returned result (check content)
   • {'image_id': '2414608', 'remove_concept': 'tree', 'add_concept': 'tree'}: Returned result (check content)


In [19]:
# Counterfactual: Qualitative Example
print("\n📝 COUNTERFACTUAL REASONING - QUALITATIVE ANALYSIS")
print("="*60)

if test_images:
    result = engine.counterfactual_reasoning(
        image_id=test_images[0],
        remove_concept='person'
    )
    print("\n✅ Removing 'person' from scene:")
    result.print_result(verbose=True)


📝 COUNTERFACTUAL REASONING - QUALITATIVE ANALYSIS

✅ Removing 'person' from scene:

🧠 ADVANCED QUERY: 4. Counterfactual Reasoning

❓ QUESTION: Counterfactual analysis for image '2414608' - if 'person' removed

🔍 REASONING STEPS (3 steps):
------------------------------------------------------------
   Step 1: 📥 [RETRIEVE]
            Load current scene for image '2414608'
            📊 0 → 10 items
            • concepts: ['watch', 'logo', 'face', 'hair', 'surfboard', 'ocean', 'hand', 'head', 'shorts', 'surfer']
            • relations: 14

   Step 2: 🧮 [COMPUTE]
            Simulate removal of concept 'person'
            📊 10 → 10 items
            • objects_removed: 0
            • relations_affected: 0

   Step 3: 📥 [RETRIEVE]
            Find images matching counterfactual scene
            📊 9,901 → 1 items
            • matching_criteria: has {'watch', 'logo', 'face', 'hair', 'surfboard', 'ocean', 'hand', 'head', 'shorts', 'surfer'}, lacks person

📊 RESULTS:
   • original_scene

---
# 📈 5. Centrality Query Evaluation

Testing centrality-based retrieval:
- Degree centrality
- PageRank centrality
- Hub images detection
- Different node filters (concept, attribute)

In [20]:
# Centrality Query: Quantitative Evaluation
centrality_metrics = EvaluationMetrics("Centrality Query")

centrality_test_cases = [
    # Degree centrality tests
    {'centrality_type': 'degree', 'node_filter': 'concept', 'top_k': 10},
    {'centrality_type': 'degree', 'node_filter': 'attribute', 'top_k': 10},
    {'centrality_type': 'degree', 'node_filter': None, 'top_k': 10},
    
    # PageRank centrality
    {'centrality_type': 'pagerank', 'node_filter': 'concept', 'top_k': 10},
    {'centrality_type': 'pagerank', 'node_filter': 'attribute', 'top_k': 10},
    
    # Hub images
    {'centrality_type': 'hub_images', 'node_filter': None, 'top_k': 10},
    {'centrality_type': 'hub_images', 'node_filter': None, 'top_k': 20},
    
    # Edge cases
    {'centrality_type': 'unknown_type', 'node_filter': 'concept', 'top_k': 5},  # Should default
    {'centrality_type': 'degree', 'node_filter': 'concept', 'top_k': 100},  # Large k
]

print(f"Running Centrality Query evaluation ({len(centrality_test_cases)} tests)...")

for i, tc in enumerate(centrality_test_cases):
    try:
        start_time = time.time()
        result = engine.centrality_query(**tc)
        elapsed = time.time() - start_time
        centrality_metrics.record(result, elapsed, tc)
        
        count = len(result.results) if isinstance(result.results, list) else 0
        print(f"   ✅ Test {i+1}: {tc['centrality_type']}/{tc['node_filter']} → {count} results ({elapsed*1000:.1f}ms)")
        
    except Exception as e:
        centrality_metrics.record_error(e, tc)
        print(f"   ❌ Test {i+1}: ERROR - {str(e)[:50]}")

centrality_metrics.print_summary()

Running Centrality Query evaluation (9 tests)...
   ✅ Test 1: degree/concept → 10 results (4.6ms)
   ✅ Test 2: degree/attribute → 10 results (1.2ms)
   ✅ Test 3: degree/None → 10 results (6.3ms)
   ✅ Test 4: pagerank/concept → 10 results (433.6ms)
   ✅ Test 5: pagerank/attribute → 10 results (1.8ms)
   ✅ Test 6: hub_images/None → 10 results (792.8ms)
   ✅ Test 7: hub_images/None → 20 results (806.6ms)
   ✅ Test 8: unknown_type/concept → 5 results (2.7ms)
   ✅ Test 9: degree/concept → 100 results (3.8ms)

📊 Centrality Query Evaluation Summary
   Total Queries: 9
   Successful: 9 (100.0%)
   Empty Results: 0
   Errors: 0
   Avg Time: 228.1 ms
   Avg Results: 20.6


In [21]:
# Centrality: Verify ranking correctness
print("\n🔍 CENTRALITY QUERY - RANKING VERIFICATION")
print("="*60)

# Check if degree centrality returns correctly sorted results
result = engine.centrality_query(centrality_type='degree', node_filter='concept', top_k=20)

scores = [r['score'] for r in result.results]
is_sorted = all(scores[i] >= scores[i+1] for i in range(len(scores)-1))

print(f"\nDegree Centrality Ranking:")
print(f"   Results correctly sorted (descending): {'✅ Yes' if is_sorted else '❌ No'}")
print(f"   Top 5 concepts:")
for r in result.results[:5]:
    print(f"      • {r.get('name', r['node_id'])}: {r['score']}")


🔍 CENTRALITY QUERY - RANKING VERIFICATION

Degree Centrality Ranking:
   Results correctly sorted (descending): ✅ Yes
   Top 5 concepts:
      • window: 4656
      • man: 4123
      • shirt: 3301
      • tree: 2939
      • wall: 2859


In [22]:
# Centrality: Hub Images Analysis
print("\n📝 HUB IMAGES - QUALITATIVE ANALYSIS")
print("="*60)

result = engine.centrality_query(centrality_type='hub_images', top_k=5)
result.print_result(verbose=True)

# Verify hub images have many objects
print("\nHub Image Details:")
for r in result.results:
    img_id = r['node_id']
    objects = engine._image_to_objects.get(img_id, set())
    concepts = set()
    for obj_id in objects:
        info = engine._get_node_info(obj_id)
        concepts.add(info.get('name', ''))
    print(f"   • Image {img_id}: {len(objects)} objects, {len(concepts)} unique concepts")


📝 HUB IMAGES - QUALITATIVE ANALYSIS

🧠 ADVANCED QUERY: 5. Centrality-Based Query

❓ QUESTION: Find most influential nodes by hub_images centrality

🔍 REASONING STEPS (3 steps):
------------------------------------------------------------
   Step 1: 📥 [RETRIEVE]
            Select concept + attribute nodes for centrality analysis
            📊 164,585 → 2,132 items
            • node_filter: global

   Step 2: 🧮 [COMPUTE]
            Calculate hub score for images (concepts × log(relations))
            📊 9,901 → 9,901 items
            • metric: hub_score

   Step 3: 📊 [AGGREGATE]
            Rank nodes and select top 5
            📊 9,901 → 5 items
            • top_k: 5

📊 RESULTS:
   1. node_id=2328327, score=187.905164, node_type=unknown, objects=42
   2. node_id=2412465, score=185.386719, node_type=unknown, objects=78
   3. node_id=2358720, score=184.180118, node_type=unknown, objects=43
   4. node_id=2377998, score=183.972533, node_type=unknown, objects=35
   5. node_id=2315861,

---
# 📋 Summary Report

In [23]:
# Compile all metrics
all_metrics = [
    chain_metrics,
    pattern_metrics,
    scene_metrics,
    counterfactual_metrics,
    centrality_metrics
]

print("\n" + "="*80)
print("📊 COMPREHENSIVE EVALUATION SUMMARY")
print("="*80)

print("\n| Query Type | Total | Success | Empty | Errors | Success Rate | Avg Time |")
print("|" + "-"*20 + "|" + "-"*7 + "|" + "-"*9 + "|" + "-"*7 + "|" + "-"*8 + "|" + "-"*14 + "|" + "-"*10 + "|")

for m in all_metrics:
    s = m.summary()
    print(f"| {s['query_type'][:18]:18} | {s['total_queries']:5} | {s['successful']:7} | {s['empty_results']:5} | {s['errors']:6} | {s['success_rate']:12} | {s['avg_time_ms']:7} ms |")


📊 COMPREHENSIVE EVALUATION SUMMARY

| Query Type | Total | Success | Empty | Errors | Success Rate | Avg Time |
|--------------------|-------|---------|-------|--------|--------------|----------|
| Chain Reasoning    |    16 |      11 |     5 |      0 | 68.8%        | 7.0     ms |
| Pattern Matching   |    13 |      11 |     2 |      0 | 84.6%        | 8.7     ms |
| Scene Comparison   |    14 |      14 |     0 |      0 | 100.0%       | 0.1     ms |
| Counterfactual Rea |    10 |      10 |     0 |      0 | 100.0%       | 86.4    ms |
| Centrality Query   |     9 |       9 |     0 |      0 | 100.0%       | 228.1   ms |


In [24]:
# Failure Case Summary
print("\n" + "="*80)
print("❌ FAILURE CASE SUMMARY")
print("="*80)

for m in all_metrics:
    if m.failure_cases:
        print(f"\n📌 {m.query_type}: {len(m.failure_cases)} failures")
        
        # Sample failure reasons
        reasons = defaultdict(int)
        for fc in m.failure_cases:
            reasons[fc.get('reason', 'unknown')] += 1
        
        for reason, count in reasons.items():
            print(f"      • {reason}: {count}")


❌ FAILURE CASE SUMMARY

📌 Chain Reasoning: 5 failures
      • empty_result: 5

📌 Pattern Matching: 2 failures
      • empty_result: 2


In [25]:
# Identified Issues and Recommendations
print("\n" + "="*80)
print("🔧 IDENTIFIED ISSUES & RECOMMENDATIONS")
print("="*80)

issues = []

# Analyze chain reasoning
chain_success = chain_metrics.successful_queries / max(1, chain_metrics.total_queries)
if chain_success < 0.5:
    issues.append({
        'query': 'Chain Reasoning',
        'issue': f'Low success rate ({chain_success:.1%})',
        'cause': 'Long chains or rare concepts often yield no results',
        'recommendation': 'Consider fallback to shorter chains or semantic similarity'
    })

# Analyze pattern matching
pattern_success = pattern_metrics.successful_queries / max(1, pattern_metrics.total_queries)
if pattern_success < 0.7:
    issues.append({
        'query': 'Pattern Matching',
        'issue': f'Moderate success rate ({pattern_success:.1%})',
        'cause': 'Complex patterns and rare concept combinations',
        'recommendation': 'Implement partial pattern matching for near-misses'
    })

# Check for slow queries
for m in all_metrics:
    avg_time = m.total_time / max(1, m.total_queries)
    if avg_time > 1.0:  # More than 1 second
        issues.append({
            'query': m.query_type,
            'issue': f'Slow average query time ({avg_time*1000:.0f}ms)',
            'cause': 'Large graph traversal or complex computation',
            'recommendation': 'Add result caching or limit traversal depth'
        })

if issues:
    for i, issue in enumerate(issues, 1):
        print(f"\n{i}. {issue['query']}")
        print(f"   Issue: {issue['issue']}")
        print(f"   Cause: {issue['cause']}")
        print(f"   Recommendation: {issue['recommendation']}")
else:
    print("\n✅ No major issues detected!")


🔧 IDENTIFIED ISSUES & RECOMMENDATIONS

✅ No major issues detected!


In [26]:
# Save evaluation results to JSON
evaluation_results = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'graph_stats': {
        'nodes': engine.graph.number_of_nodes(),
        'edges': engine.graph.number_of_edges(),
        'concepts': len(engine._concept_nodes),
        'attributes': len(engine._attribute_nodes),
        'images': len(engine._image_to_objects)
    },
    'metrics': [m.summary() for m in all_metrics],
    'issues': issues
}

# Save to file
output_path = project_root / 'reasoning' / 'evaluation_results.json'
with open(output_path, 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f"\n✅ Results saved to: {output_path}")


✅ Results saved to: d:\Giáo trình 20251\IT3930E - Project III\hybrid_multimodal_retrieval\reasoning\evaluation_results.json


---
# 🎯 Key Failure Cases Identified

## 1. Chain Reasoning Failures
- **Long chains (3+ hops)**: Very few paths exist in the graph
- **Attribute constraints**: Narrows results too aggressively
- **Rare concepts**: Starting with rare concepts yields no traversable paths

## 2. Pattern Matching Failures
- **Rare concept combinations**: Patterns with rare nodes don't exist
- **Complex patterns (3+ edges)**: Exponential reduction in matches
- **Specific relations**: Some relation types are rare in the graph

## 3. Scene Comparison Edge Cases
- **Non-existent images**: Need better error handling
- **Single-object images**: Similarity metrics may be skewed
- **Completely disjoint scenes**: 0% similarity is valid but may need interpretation

## 4. Counterfactual Reasoning Edge Cases
- **Removing absent concepts**: No impact to analyze
- **Adding concepts with no typical relations**: Empty predictions
- **Non-existent images**: Need graceful degradation

## 5. Centrality Query Issues
- **PageRank convergence**: May fail on disconnected subgraphs
- **Instance node analysis**: Very slow on large graphs
- **Unknown centrality types**: Should have explicit error handling

In [27]:
print("\n" + "="*80)
print("🎉 EVALUATION COMPLETE!")
print("="*80)
print("\nKey findings:")
print(f"   • Total queries executed: {sum(m.total_queries for m in all_metrics)}")
print(f"   • Overall success rate: {sum(m.successful_queries for m in all_metrics) / sum(m.total_queries for m in all_metrics):.1%}")
print(f"   • Total failures detected: {sum(len(m.failure_cases) for m in all_metrics)}")
print(f"   • Issues requiring attention: {len(issues)}")


🎉 EVALUATION COMPLETE!

Key findings:
   • Total queries executed: 62
   • Overall success rate: 88.7%
   • Total failures detected: 7
   • Issues requiring attention: 0


---
# 🔬 Detailed Failure Case Investigation

Deep analysis of identified failure cases with root cause analysis.

In [28]:
# Deep Investigation: Chain Reasoning Failures
print("🔬 CHAIN REASONING - DETAILED FAILURE INVESTIGATION")
print("="*80)

# Test specific failure scenarios
failure_tests = [
    # Failure Type 1: 2-hop chains with spatial relations
    {
        "name": "2-hop with 'to the left of'",
        "params": {"start_concept": "person", "chain": [
            {"relation": "wearing", "concept": "shirt"},
            {"relation": "to the left of", "concept": "tree"}
        ]},
        "expected_issue": "Spatial relations between objects of different types are rare"
    },
    # Failure Type 2: Chain with attribute constraint
    {
        "name": "Attribute constraint too specific",
        "params": {"start_concept": "car", "chain": [
            {"relation": "on", "concept": "road", "attribute": "black"}
        ]},
        "expected_issue": "Attribute 'black' rarely applied to 'road'"
    },
    # Failure Type 3: Long chain traversal
    {
        "name": "3-hop chain",
        "params": {"start_concept": "person", "chain": [
            {"relation": "wearing", "concept": "shirt"},
            {"relation": "to the left of", "concept": "table"},
            {"relation": "in", "concept": "room"}
        ]},
        "expected_issue": "Path length exponentially reduces matches"
    },
]

for test in failure_tests:
    print(f"\n📌 Testing: {test['name']}")
    print(f"   Expected Issue: {test['expected_issue']}")
    
    result = engine.chain_reasoning(**test['params'], limit=10)
    
    # Analyze each step to find where the chain breaks
    print(f"   Reasoning Step Analysis:")
    for step in result.reasoning_steps:
        reduction = "N/A"
        if step.input_size > 0:
            if step.output_size == 0:
                reduction = "BREAKDOWN ❌"
            else:
                reduction = f"{step.output_size/step.input_size*100:.1f}% retained"
        print(f"      Step {step.step_number}: {step.operation} → {step.input_size} → {step.output_size} ({reduction})")
    
    print(f"   Final Results: {len(result.results) if isinstance(result.results, list) else 0}")

🔬 CHAIN REASONING - DETAILED FAILURE INVESTIGATION

📌 Testing: 2-hop with 'to the left of'
   Expected Issue: Spatial relations between objects of different types are rare
   Reasoning Step Analysis:
      Step 1: RETRIEVE → 0 → 2786 (N/A)
      Step 2: TRAVERSE → 500 → 148 (29.6% retained)
      Step 3: FILTER → 148 → 29 (19.6% retained)
      Step 4: TRAVERSE → 29 → 108 (372.4% retained)
      Step 5: FILTER → 108 → 0 (BREAKDOWN ❌)
      Step 6: AGGREGATE → 0 → 0 (N/A)
   Final Results: 0

📌 Testing: Attribute constraint too specific
   Expected Issue: Attribute 'black' rarely applied to 'road'
   Reasoning Step Analysis:
      Step 1: RETRIEVE → 0 → 1497 (N/A)
      Step 2: TRAVERSE → 500 → 133 (26.6% retained)
      Step 3: FILTER → 133 → 53 (39.8% retained)
      Step 4: FILTER → 53 → 0 (BREAKDOWN ❌)
      Step 5: AGGREGATE → 0 → 0 (N/A)
   Final Results: 0

📌 Testing: 3-hop chain
   Expected Issue: Path length exponentially reduces matches
   Reasoning Step Analysis:
      Step 1

In [29]:
# Deep Investigation: Pattern Matching Failures
print("🔬 PATTERN MATCHING - DETAILED FAILURE INVESTIGATION")
print("="*80)

# Investigate why certain patterns fail
failure_patterns = [
    {
        "name": "Rare concept combination (person + pant)",
        "nodes": ["person", "shirt", "pant"],
        "edges": [("person", "wearing", "shirt"), ("person", "wearing", "pant")],
        "expected_issue": "Concept 'pant' may be labeled differently (pants, trousers)"
    },
    {
        "name": "Spatial relation between unrelated objects",
        "nodes": ["airplane", "bird"],
        "edges": [("airplane", "above", "bird")],
        "expected_issue": "These objects rarely appear in same image"
    }
]

for pattern in failure_patterns:
    print(f"\n📌 Pattern: {pattern['name']}")
    print(f"   Expected Issue: {pattern['expected_issue']}")
    
    # Check concept existence
    print(f"   Concept Check:")
    for node in pattern['nodes']:
        concept_id = f"Concept:{node}"
        exists = concept_id in engine._concept_nodes
        count = concept_counts.get(node, 0)
        print(f"      • '{node}': exists={exists}, instances={count}")
    
    # Check relation existence
    print(f"   Relation Check:")
    for edge in pattern['edges']:
        rel = edge[1]
        rel_count = relation_counts.get(rel, 0)
        print(f"      • '{rel}': {rel_count} edges in graph")
    
    # Try the query
    result = engine.pattern_matching(
        pattern_nodes=pattern['nodes'],
        pattern_edges=pattern['edges'],
        limit=5
    )
    
    print(f"   Final Results: {len(result.results) if isinstance(result.results, list) else 0}")
    
    # Check for similar concepts
    if 'pant' in pattern['nodes']:
        print(f"\\n   Checking for similar concepts:")
        for concept in ['pants', 'trousers', 'jeans', 'shorts']:
            count = concept_counts.get(concept, 0)
            if count > 0:
                print(f"      • '{concept}': {count} instances")

🔬 PATTERN MATCHING - DETAILED FAILURE INVESTIGATION

📌 Pattern: Rare concept combination (person + pant)
   Expected Issue: Concept 'pant' may be labeled differently (pants, trousers)
   Concept Check:
      • 'person': exists=True, instances=2786
      • 'shirt': exists=True, instances=3301
      • 'pant': exists=False, instances=0
   Relation Check:
      • 'wearing': 6808 edges in graph
      • 'wearing': 6808 edges in graph
   Final Results: 0
\n   Checking for similar concepts:
      • 'pants': 1259 instances
      • 'jeans': 477 instances
      • 'shorts': 742 instances

📌 Pattern: Spatial relation between unrelated objects
   Expected Issue: These objects rarely appear in same image
   Concept Check:
      • 'airplane': exists=True, instances=436
      • 'bird': exists=True, instances=507
   Relation Check:
      • 'above': 1117 edges in graph
   Final Results: 0


In [30]:
# Stress Tests: Edge Cases and Boundary Conditions
print("🔬 EDGE CASE & STRESS TESTS")
print("="*80)

edge_case_results = []

# Test 1: Empty/Invalid inputs
print("\n📌 Test 1: Empty/Invalid Inputs")
try:
    result = engine.chain_reasoning(start_concept="", chain=[])
    edge_case_results.append(("Empty chain", "Passed", len(result.results)))
    print(f"   • Empty chain: Returned {len(result.results)} results")
except Exception as e:
    edge_case_results.append(("Empty chain", "Error", str(e)[:50]))
    print(f"   • Empty chain: Error - {str(e)[:50]}")

# Test 2: Non-existent concept
print("\n📌 Test 2: Non-existent Concepts")
try:
    result = engine.chain_reasoning(
        start_concept="xyznonexistent123",
        chain=[{"relation": "wearing", "concept": "shirt"}]
    )
    edge_case_results.append(("Non-existent start", "Passed", len(result.results)))
    print(f"   • Non-existent start concept: Returned {len(result.results)} results")
except Exception as e:
    edge_case_results.append(("Non-existent start", "Error", str(e)[:50]))
    print(f"   • Non-existent start concept: Error - {str(e)[:50]}")

# Test 3: Very large limit
print("\n📌 Test 3: Very Large Limit")
start_time = time.time()
result = engine.chain_reasoning(
    start_concept="person",
    chain=[{"relation": "wearing"}],
    limit=10000
)
elapsed = time.time() - start_time
edge_case_results.append(("Large limit", "Passed", f"{len(result.results)} in {elapsed:.2f}s"))
print(f"   • Limit=10000: Returned {len(result.results)} results in {elapsed:.2f}s")

# Test 4: Same image comparison
print("\n📌 Test 4: Same Image Comparison")
sample_img = list(engine._image_to_objects.keys())[0]
result = engine.scene_comparison(sample_img, sample_img)
sim = result.results.get('comparison', {}).get('overall_similarity', 'N/A')
edge_case_results.append(("Same image comparison", "Passed", f"similarity={sim}"))
print(f"   • Comparing image with itself: similarity={sim}")
if sim != 1.0:
    print(f"   ⚠️ ISSUE: Same image should have similarity 1.0!")

# Test 5: Counterfactual with non-existent image
print("\n📌 Test 5: Non-existent Image Handling")
try:
    result = engine.counterfactual_reasoning(
        image_id="INVALID_IMAGE_12345",
        remove_concept="person"
    )
    original = result.results.get('original_scene', {})
    edge_case_results.append(("Invalid image", "Passed", f"concepts={len(original.get('concepts', []))}"))
    print(f"   • Non-existent image: Returned scene with {len(original.get('concepts', []))} concepts")
except Exception as e:
    edge_case_results.append(("Invalid image", "Error", str(e)[:50]))
    print(f"   • Non-existent image: Error - {str(e)[:50]}")

print("\n📊 Edge Case Summary:")
print("-"*60)
for name, status, detail in edge_case_results:
    icon = "✅" if status == "Passed" else "❌"
    print(f"   {icon} {name}: {detail}")

🔬 EDGE CASE & STRESS TESTS

📌 Test 1: Empty/Invalid Inputs
   • Empty chain: Returned 0 results

📌 Test 2: Non-existent Concepts
   • Non-existent start concept: Returned 0 results

📌 Test 3: Very Large Limit
   • Limit=10000: Returned 97 results in 0.01s

📌 Test 4: Same Image Comparison
   • Comparing image with itself: similarity=1.0

📌 Test 5: Non-existent Image Handling
   • Non-existent image: Returned scene with 0 concepts

📊 Edge Case Summary:
------------------------------------------------------------
   ✅ Empty chain: 0
   ✅ Non-existent start: 0
   ✅ Large limit: 97 in 0.01s
   ✅ Same image comparison: similarity=1.0
   ✅ Invalid image: concepts=0


## 📋 Main Failure Cases Summary

Based on the detailed investigation, here are the **main failure cases** identified for each query type:

### 1. Chain Reasoning Failures

| Failure Type | Root Cause | Frequency | Recommendation |
|--------------|------------|-----------|----------------|
| **Long chains (3+ hops)** | Path sparsity - few valid paths exist | High | Limit to 2-hop chains or use semantic fallback |
| **Attribute constraints** | Overly specific filtering | Medium | Make attributes optional or use fuzzy matching |
| **Spatial relations in chains** | Objects rarely connected spatially | Medium | Pre-compute spatial neighborhoods |

### 2. Pattern Matching Failures

| Failure Type | Root Cause | Frequency | Recommendation |
|--------------|------------|-----------|----------------|
| **Rare concept combinations** | Low co-occurrence in dataset | High | Add synonym expansion (e.g., pant→pants) |
| **Unusual relation types** | Some relations have few edges | Medium | Fall back to semantic similarity |
| **Complex patterns (3+ edges)** | Exponential filtering effect | Low | Use partial matching with confidence scores |

### 3. Scene Comparison Edge Cases

| Edge Case | Behavior | Status |
|-----------|----------|--------|
| **Same image comparison** | Should return similarity=1.0 | ⚠️ Check implementation |
| **Non-existent image** | Returns empty scene gracefully | ✅ OK |
| **Disjoint scenes** | Returns similarity=0.0 | ✅ OK |

### 4. Counterfactual Reasoning Edge Cases

| Edge Case | Behavior | Status |
|-----------|----------|--------|
| **Remove missing concept** | Returns unchanged scene | ✅ OK |
| **Non-existent image** | Returns empty scene | ✅ OK |
| **Add rare concept** | May have empty predictions | ⚠️ Expected |

### 5. Centrality Query Notes

| Observation | Status |
|-------------|--------|
| **PageRank computation** | Slow on full graph (~400ms) | Consider caching |
| **Hub images** | Correctly identifies dense images | ✅ OK |
| **Unknown centrality type** | Falls back to degree | ✅ OK |